# Profilage Automatique des données

Ce notebook génère automatiquement des rapports de qualité des données en utilisant :
- **Evidently** : Rapport de qualité détaillé avec métriques avancées
- **Sweetviz** : Analyse exploratoire visuelle interactive

Les données sont chargées directement depuis la base PostgreSQL.

In [1]:
# Import des bibliothèques
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os
from evidently.legacy.report import Report
from evidently.legacy.metric_preset import DataQualityPreset
import sweetviz as sv
from datetime import datetime

In [2]:
# Configuration de la connexion à la base de données
load_dotenv('../.env')  # Charger le fichier .env du répertoire parent

DB_HOST = os.getenv('DB_HOST', 'host.docker.internal')
DB_PORT = os.getenv('DB_PORT', '5432')
DB_NAME = os.getenv('POSTGRES_DB', 'games_db')
DB_USER = os.getenv('POSTGRES_USER', 'postgres')
DB_PASSWORD = os.getenv('POSTGRES_PASSWORD', 'postgres')

CONNECTION_STRING = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

print(f"Connexion à : {DB_HOST}:{DB_PORT}/{DB_NAME} avec utilisateur {DB_USER}")

Connexion à : postgres:5432/games_db avec utilisateur postgres


In [3]:
# Chargement des données depuis PostgreSQL
try:
    engine = create_engine(CONNECTION_STRING)
    
    # Charger toutes les données de la table games_brut
    df = pd.read_sql("SELECT * FROM games_brut", engine)
    
    print(f"Données chargées avec succès !")
    print(f"Shape : {df.shape[0]} lignes, {df.shape[1]} colonnes")
    print(f"Colonnes : {list(df.columns)}")
    
    # Aperçu des données
    display(df.head())
    
except Exception as e:
    print(f"Erreur lors du chargement des données : {e}")
    df = None

Données chargées avec succès !
Shape : 122611 lignes, 40 colonnes
Colonnes : ['app_id', 'name', 'release_date', 'estimated_owners', 'peak_ccu', 'required_age', 'price', 'discount', 'dlc_count', 'about_the_game', 'supported_languages', 'full_audio_languages', 'reviews', 'header_image', 'website', 'support_url', 'support_email', 'windows', 'mac', 'linux', 'metacritic_score', 'metacritic_url', 'user_score', 'positive', 'negative', 'score_rank', 'achievements', 'recommendations', 'notes', 'average_playtime_forever', 'average_playtime_two_weeks', 'median_playtime_forever', 'median_playtime_two_weeks', 'developers', 'publishers', 'categories', 'genres', 'tags', 'screenshots', 'movies']


,app_id,name,release_date,estimated_owners,peak_ccu,required_age,price,discount,dlc_count,about_the_game,...,average_playtime_two_weeks,median_playtime_forever,median_playtime_two_weeks,developers,publishers,categories,genres,tags,screenshots,movies
0,2539430,Black Dragon Mage Playtest,2023-08-01,0 - 0,0,0,0.00,0,0,None,...,0,0,0,None,None,None,None,None,https://shared.akamai.steamstatic.com/store_it...,None
1,496350,Supipara - Chapter 1 Spring Has Come!,2016-07-29,0 - 20000,0,0,5.24,65,0,"Springtime, April: when the cherry trees come ...",...,0,8,0,minori,MangaGamer,"Single-player,Steam Trading Cards,Steam Cloud,...",Adventure,"Adventure,Visual Novel,Anime,Cute",https://shared.akamai.steamstatic.com/store_it...,None
2,1034400,Mystery Solitaire The Black Raven,2019-05-06,0 - 20000,0,0,4.99,0,0,"Immerse yourself in the most beloved, mystical...",...,0,0,0,Somer Games,8floor,"Single-player,Family Sharing",Casual,"Casual,Card Game,Solitaire,Puzzle,Hidden Objec...",https://shared.akamai.steamstatic.com/store_it...,None
3,3292190,버튜버 파라노이아 - Vtuber Paranoia,2024-10-31,0 - 20000,1,0,8.99,0,1,"synopsis 'Hello, I'm Hiyoro, a new YouTuber!' ...",...,0,0,0,유진게임즈,유진게임즈,"Single-player,Steam Achievements,Family Sharing","Casual,Indie,Simulation",None,https://shared.akamai.steamstatic.com/store_it...,None
4,3631080,Maze Quest VR,2025-04-24,0 - 20000,0,0,4.99,0,0,Its not just a Maze; its a Quest! Enter the ca...,...,0,0,0,Reality Expanded LLC,Reality Expanded LLC,"Single-player,VR Only,Steam Leaderboards,Famil...","Action,Early Access",None,https://shared.akamai.steamstatic.com/store_it...,None


In [4]:
# Génération du rapport Evidently
if df is not None:
    try:
        print("Génération du rapport Evidently...")
        
        # Créer le rapport de qualité des données
        report = Report(metrics=[
            DataQualityPreset(),
        ])
        
        # Exécuter le rapport
        report.run(current_data=df, reference_data=None)
        
        # Sauvegarder avec timestamp
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        evidently_filename = f"../reports/evidently_data_quality_report_{timestamp}.html"
        report.save_html(evidently_filename)
        
        print(f"Evidently : {evidently_filename}")
        
    except Exception as e:
        print(f"Erreur Evidently : {e}")
else:
    print("Pas de données à analyser")

Génération du rapport Evidently...
Evidently : ../reports/evidently_data_quality_report_20260203_132010.html


In [5]:
# Génération du rapport Sweetviz
if df is not None:
    try:
        print("Génération du rapport Sweetviz")
        
        # Analyser les données avec Sweetviz
        sv_report = sv.analyze(df)
        
        # Sauvegarder avec timestamp
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        sweetviz_filename = f"../reports/sweetviz_report_{timestamp}.html"
        sv_report.show_html(sweetviz_filename, open_browser=False)
        
        print(f"Sweetviz : {sweetviz_filename}")
        
    except Exception as e:
        print(f"Erreur Sweetviz : {e}")
else:
    print("Pas de données à analyser")

Génération du rapport Sweetviz


                                             |          | [  0%]   00:00 -> (? left)

Report ../reports/sweetviz_report_20260203_132031.html was generated.
Sweetviz : ../reports/sweetviz_report_20260203_132031.html
